# Análisis de Rutas Aéreas con Algoritmos de Grafos

Este proyecto implementa un sistema para analizar datos de vuelos usando algoritmos de grafos. Utiliza los archivos `airports.dat` y `routes.dat` del dataset OpenFlights para construir un grafo dirigido donde los aeropuertos son nodos y las rutas son aristas.

## 1. Configuración Inicial y Funciones Auxiliares

Esta sección incluye las importaciones necesarias y funciones de ayuda, como el cálculo de distancia geográfica.

In [ ]:
import csv
import math
from collections import defaultdict
import heapq

def calcular_distancia_haversine(latitud1, longitud1, latitud2, longitud2):
    RADIO_TERRESTRE = 6371

    latitud1_radianes = math.radians(latitud1)
    longitud1_radianes = math.radians(longitud1)
    latitud2_radianes = math.radians(latitud2)
    longitud2_radianes = math.radians(longitud2)

    diferencia_longitud = longitud2_radianes - longitud1_radianes
    diferencia_latitud = latitud2_radianes - latitud1_radianes

    a_valor = math.sin(diferencia_latitud / 2)**2 + math.cos(latitud1_radianes) * math.cos(latitud2_radianes) * math.sin(diferencia_longitud / 2)**2
    c_valor = 2 * math.atan2(math.sqrt(a_valor), math.sqrt(1 - a_valor))

    distancia_calculada = RADIO_TERRESTRE * c_valor
    return distancia_calculada

## 2. Carga y Limpieza de Datos

Esta sección se encarga de leer los archivos `airports.dat` y `routes.dat`, limpiar los datos y construir las estructuras de datos principales (`aeropuertos` y `grafo`).

In [ ]:
def cargar_datos_aeropuertos(ruta_archivo = 'airports.dat'):
    mapa_aeropuertos = {}
    with open(ruta_archivo, 'r', encoding='utf-8') as archivo:
        csv_reader = csv.reader(archivo)
        for indice, fila in enumerate(csv_reader):
            if len(fila) < 7:
                continue
            try:
                id_aeropuerto = fila[0]
                nombre = fila[1]
                ciudad = fila[2]
                pais = fila[3]
                codigo_iata = fila[4] if fila[4] != '\\N' else None
                latitud = float(fila[6])
                longitud = float(fila[7])

                if not id_aeropuerto or id_aeropuerto == '\\N':
                    continue

                mapa_aeropuertos[id_aeropuerto] = {
                    "nombre": nombre,
                    "iata": codigo_iata,
                    "lat": latitud,
                    "lon": longitud,
                    "ciudad": ciudad,
                    "pais": pais
                }
            except (ValueError, IndexError):
                continue
    print(f"Cargados {len(mapa_aeropuertos)} aeropuertos.")
    return mapa_aeropuertos

def cargar_datos_rutas(ruta_archivo = 'routes.dat', mapa_aeropuertos = None):
    if mapa_aeropuertos is None:
        mapa_aeropuertos = {}

    grafo_vuelos = defaultdict(list)
    rutas_unicas = set()
    conteo_rutas_invalidas = 0
    conteo_rutas_validas = 0

    with open(ruta_archivo, 'r', encoding='utf-8') as archivo:
        csv_reader = csv.reader(archivo)
        for indice, fila in enumerate(csv_reader):
            if len(fila) < 6:
                continue
            try:
                id_origen = fila[3]
                id_destino = fila[5]

                if not id_origen or id_origen == '\\N' or \
                   not id_destino or id_destino == '\\N':
                    conteo_rutas_invalidas += 1
                    continue

                if id_origen not in mapa_aeropuertos or \
                   id_destino not in mapa_aeropuertos:
                    conteo_rutas_invalidas += 1
                    continue

                tupla_ruta = (id_origen, id_destino)
                if tupla_ruta in rutas_unicas:
                    continue
                rutas_unicas.add(tupla_ruta)

                grafo_vuelos[id_origen].append(id_destino)
                conteo_rutas_validas += 1

            except (ValueError, IndexError):
                conteo_rutas_invalidas += 1
                continue
    print(f"Cargadas {conteo_rutas_validas} rutas. ({conteo_rutas_invalidas} rutas inválidas/ignoradas)")
    return grafo_vuelos

todos_los_aeropuertos = cargar_datos_aeropuertos()
grafo_de_vuelos = cargar_datos_rutas(mapa_aeropuertos=todos_los_aeropuertos)

Cargados 7698 aeropuertos.
Cargadas 36907 rutas. (892 rutas inválidas/ignoradas)


## 3. Reto 1: Alcance Personalizado (BFS)

Esta sección implementa un algoritmo de Búsqueda en Amplitud (BFS) para determinar cuántos aeropuertos distintos son alcanzables desde un aeropuerto de origen con un máximo de 3 escalas.

In [ ]:
def encontrar_aeropuerto_por_nombre_o_iata(consulta, mapa_aeropuertos):
    consulta_normalizada = consulta.strip().upper()
    for id_aeropuerto, datos_aeropuerto in mapa_aeropuertos.items():
        nombre_aeropuerto = str(datos_aeropuerto.get('nombre', '')).upper()
        codigo_iata = str(datos_aeropuerto.get('iata', '')).upper()
        if consulta_normalizada == nombre_aeropuerto or consulta_normalizada == codigo_iata:
            return id_aeropuerto
    return None

def calcular_alcance_bfs(id_origen, grafo, max_escalas = 3):
    if id_origen not in grafo:
        return 0, set()

    visitados = {id_origen}
    cola = [(id_origen, 0)]
    aeropuertos_alcanzables = {id_origen}

    while cola:
        id_aeropuerto_actual, escalas_actuales = cola.pop(0)

        if escalas_actuales < max_escalas:
            for id_vecino in grafo[id_aeropuerto_actual]:
                if id_vecino not in visitados:
                    visitados.add(id_vecino)
                    aeropuertos_alcanzables.add(id_vecino)
                    cola.append((id_vecino, escalas_actuales + 1))

    return len(aeropuertos_alcanzables), aeropuertos_alcanzables

def reto1_alcance_personalizado(mapa_aeropuertos, grafo_vuelos):
    print("\n--- Reto 1: Alcance Personalizado ---")
    consulta_usuario = input("Ingrese el nombre o código IATA de un aeropuerto: ")

    id_aeropuerto_origen = encontrar_aeropuerto_por_nombre_o_iata(consulta_usuario, mapa_aeropuertos)

    if id_aeropuerto_origen:
        nombre_aeropuerto_origen = mapa_aeropuertos[id_aeropuerto_origen]['nombre']
        cantidad_alcanzables, _ = calcular_alcance_bfs(id_aeropuerto_origen, grafo_vuelos)
        print(f"Desde '{nombre_aeropuerto_origen}', se pueden alcanzar {cantidad_alcanzables} aeropuertos distintos con un máximo de 3 escalas.")
    else:
        print(f"No se encontró ningún aeropuerto con '{consulta_usuario}'.")

## 4. Reto 2: Grupos y Aislamiento Aéreo (Kosaraju)

Esta sección implementa el algoritmo de Kosaraju para encontrar los componentes fuertemente conexas (Strongly Connected Components - SCCs) en el grafo, lo que nos permite identificar grupos de aeropuertos aislados.

In [ ]:
def dfs_orden_finalizacion(nodo_actual, grafo, visitados, pila):
    visitados.add(nodo_actual)
    for vecino in grafo[nodo_actual]:
        if vecino not in visitados:
            dfs_orden_finalizacion(vecino, grafo, visitados, pila)
    pila.append(nodo_actual)

def construir_grafo_transpuesto(grafo):
    grafo_transpuesto = defaultdict(list)
    for nodo_u in grafo:
        for nodo_v in grafo[nodo_u]:
            grafo_transpuesto[nodo_v].append(nodo_u)
    for nodo_u in grafo:
        if nodo_u not in grafo_transpuesto:
            grafo_transpuesto[nodo_u] = []
    return grafo_transpuesto

def dfs_scc(nodo_actual, grafo_transpuesto, visitados, componente_actual):
    visitados.add(nodo_actual)
    componente_actual.append(nodo_actual)
    for vecino in grafo_transpuesto[nodo_actual]:
        if vecino not in visitados:
            dfs_scc(vecino, grafo_transpuesto, visitados, componente_actual)

def kosaraju_scc(grafo):
    pila = []
    visitados = set()

    for nodo in grafo:
        if nodo not in visitados:
            dfs_orden_finalizacion(nodo, grafo, visitados, pila)

    for id_aeropuerto in todos_los_aeropuertos.keys():
        if id_aeropuerto not in visitados:
            dfs_orden_finalizacion(id_aeropuerto, grafo, visitados, pila)

    grafo_transpuesto = construir_grafo_transpuesto(grafo)

    visitados.clear()
    componentes_fuertemente_conexas = []
    while pila:
        nodo_u = pila.pop()
        if nodo_u not in visitados:
            componente_actual = []
            dfs_scc(nodo_u, grafo_transpuesto, visitados, componente_actual)
            componentes_fuertemente_conexas.append(componente_actual)

    return componentes_fuertemente_conexas

def reto2_grupos_aislados(mapa_aeropuertos, grafo_vuelos):
    print("\n--- Reto 2: Grupos y Aislamiento Aéreo ---")
    componentes_fuertemente_conexas = kosaraju_scc(grafo_vuelos)

    num_componentes = len(componentes_fuertemente_conexas)
    tamano_max_componente = 0
    if componentes_fuertemente_conexas:
        tamano_max_componente = max(len(componente) for componente in componentes_fuertemente_conexas)

    print(f"Cantidad total de grupos aislados (SCCs): {num_componentes}")
    print(f"Tamaño del grupo más grande: {tamano_max_componente} aeropuertos")

## 5. Reto 3: La Máxima Eficiencia (Diámetro) (Dijkstra)

Esta sección se enfoca en encontrar el par de aeropuertos cuya distancia mínima total sea la mayor en el grafo, utilizando el algoritmo de Dijkstra y la fórmula de Haversine para las distancias entre aeropuertos conectados.

In [ ]:
def dijkstra(id_origen, grafo, mapa_aeropuertos):
    distancias = {nodo: float('inf') for nodo in mapa_aeropuertos.keys()}
    distancias[id_origen] = 0
    cola_prioridad = [(0, id_origen)]

    while cola_prioridad:
        distancia_actual, id_aeropuerto_actual = heapq.heappop(cola_prioridad)

        if distancia_actual > distancias[id_aeropuerto_actual]:
            continue

        for id_vecino in grafo[id_aeropuerto_actual]:
            if id_aeropuerto_actual in mapa_aeropuertos and id_vecino in mapa_aeropuertos:
                lat1 = float(mapa_aeropuertos[id_aeropuerto_actual]['lat'])
                lon1 = float(mapa_aeropuertos[id_aeropuerto_actual]['lon'])
                lat2 = float(mapa_aeropuertos[id_vecino]['lat'])
                lon2 = float(mapa_aeropuertos[id_vecino]['lon'])
                peso = calcular_distancia_haversine(lat1, lon1, lat2, lon2)

                if distancia_actual + peso < distancias[id_vecino]:
                    distancias[id_vecino] = distancia_actual + peso
                    heapq.heappush(cola_prioridad, (distancias[id_vecino], id_vecino))
    return distancias

def reto3_maxima_eficiencia(mapa_aeropuertos, grafo_vuelos):
    print("\n--- Reto 3: Máxima Eficiencia (Diámetro) ---")

    max_distancia_total = 0.0
    origen_mas_lejano = None
    destino_mas_lejano = None

    aeropuertos_con_rutas_salida = [id_aeropuerto for id_aeropuerto in mapa_aeropuertos.keys() if id_aeropuerto in grafo_vuelos]

    print(f"Calculando distancias mínimas para {len(aeropuertos_con_rutas_salida)} aeropuertos de origen. Esto puede tomar un tiempo...")

    for i, id_origen in enumerate(aeropuertos_con_rutas_salida):
        if id_origen not in grafo_vuelos or not grafo_vuelos[id_origen]:
            continue

        distancias_desde_origen = dijkstra(id_origen, grafo_vuelos, mapa_aeropuertos)

        for id_destino, distancia_actual_recorrida in distancias_desde_origen.items():
            if distancia_actual_recorrida != float('inf') and distancia_actual_recorrida > max_distancia_total and id_origen != id_destino:
                max_distancia_total = distancia_actual_recorrida
                origen_mas_lejano = id_origen
                destino_mas_lejano = id_destino

        if (i + 1) % 100 == 0:
            print(f"  Progreso: {i + 1}/{len(aeropuertos_con_rutas_salida)} aeropuertos procesados.")

    if origen_mas_lejano and destino_mas_lejano:
        nombre_origen = mapa_aeropuertos[origen_mas_lejano]['nombre']
        nombre_destino = mapa_aeropuertos[destino_mas_lejano]['nombre']
        print(f"\nPar de aeropuertos con la distancia mínima total más larga:")
        print(f"  Origen: {nombre_origen} (ID: {origen_mas_lejano})")
        print(f"  Destino: {nombre_destino} (ID: {destino_mas_lejano})")
        print(f"  Distancia total mínima más larga: {max_distancia_total:.2f} km")
    else:
        print("No se pudo encontrar un par de aeropuertos con distancias finitas.\n" \
              "Esto puede ocurrir si el grafo es desconexo o si no hay rutas válidas.")

## 6. Menú Principal (Main)

Esta sección provee un menú interactivo en consola para ejecutar cada uno de los retos implementados.

In [ ]:
def mostrar_menu():
    print("\n--- MENÚ PRINCIPAL ---")
    print("1. Ejecutar Reto 1: Alcance Personalizado")
    print("2. Ejecutar Reto 2: Grupos y Aislamiento Aéreo")
    print("3. Ejecutar Reto 3: Máxima Eficiencia (Diámetro)")
    print("4. Salir")

def main():
    while True:
        mostrar_menu()
        opcion = input("Seleccione una opción: ")

        if opcion == '1':
            reto1_alcance_personalizado(todos_los_aeropuertos, grafo_de_vuelos)
        elif opcion == '2':
            reto2_grupos_aislados(todos_los_aeropuertos, grafo_de_vuelos)
        elif opcion == '3':
            reto3_maxima_eficiencia(todos_los_aeropuertos, grafo_de_vuelos)
        elif opcion == '4':
            print("¡Hasta luego!")
            break
        else:
            print("Opción no válida. Por favor, intente de nuevo.")

if __name__ == '__main__':
    main()


--- MENÚ PRINCIPAL ---
1. Ejecutar Reto 1: Alcance Personalizado
2. Ejecutar Reto 2: Grupos y Aislamiento Aéreo
3. Ejecutar Reto 3: Máxima Eficiencia (Diámetro)
4. Salir

--- Reto 1: Alcance Personalizado ---
Desde 'Viru Viru International Airport', se pueden alcanzar 1917 aeropuertos distintos con un máximo de 3 escalas.

--- MENÚ PRINCIPAL ---
1. Ejecutar Reto 1: Alcance Personalizado
2. Ejecutar Reto 2: Grupos y Aislamiento Aéreo
3. Ejecutar Reto 3: Máxima Eficiencia (Diámetro)
4. Salir

--- Reto 1: Alcance Personalizado ---
Desde 'Jorge Wilsterman International Airport', se pueden alcanzar 370 aeropuertos distintos con un máximo de 3 escalas.

--- MENÚ PRINCIPAL ---
1. Ejecutar Reto 1: Alcance Personalizado
2. Ejecutar Reto 2: Grupos y Aislamiento Aéreo
3. Ejecutar Reto 3: Máxima Eficiencia (Diámetro)
4. Salir

--- Reto 3: Máxima Eficiencia (Diámetro) ---
Calculando distancias mínimas para 3199 aeropuertos de origen. Esto puede tomar un tiempo...
  Progreso: 100/3199 aeropuertos p

**Análisis**
Sí. Tu código construye principalmente **un grafo dirigido de aeropuertos y rutas aéreas**, y luego aplica distintos algoritmos sobre ese mismo grafo.

Te dejo un esquema conceptual sencillo de cómo se “forma” el grafo y cómo cada reto lo utiliza.

---

# 1. Cómo se construye el grafo principal

## Entrada de datos

Tienes dos archivos:

* `airports.dat`
  → contiene los aeropuertos

* `routes.dat`
  → contiene las rutas de vuelos

---

# 2. Estructura de nodos (vértices)

Cada aeropuerto se convierte en un **nodo** del grafo.

Ejemplo:

| ID | Aeropuerto |
| -- | ---------- |
| 1  | La Paz     |
| 2  | Cochabamba |
| 3  | Santa Cruz |

Visualmente:

```text
(1) La Paz
(2) Cochabamba
(3) Santa Cruz
```

---

# 3. Estructura de aristas

Cada ruta aérea se convierte en una **arista dirigida**.

Si existe un vuelo:

```text
La Paz → Cochabamba
```

entonces el grafo guarda:

```python
grafo_vuelos["1"].append("2")
```

---

# 4. Forma real del grafo

Tu estructura es:

```python
defaultdict(list)
```

Entonces queda algo así:

```python
{
    "1": ["2", "3"],
    "2": ["3"],
    "3": ["1"]
}
```

Eso significa:

* Desde 1 puedes ir a 2 y 3
* Desde 2 puedes ir a 3
* Desde 3 puedes ir a 1

---

# 5. Representación visual

El grafo sería:

```text
        ┌──────────┐
        ↓          │
(1) ───→ (2) ───→ (3)
 ↑                 │
 └─────────────────┘
```

Como las rutas tienen dirección, el grafo es:

* dirigido
* no ponderado (por ahora)
* de adyacencia

---

# 6. Cómo se crea paso a paso

## Paso A — cargar aeropuertos

Función:

```python
cargar_datos_aeropuertos()
```

Crea:

```python
dic_aeropuertos[id] = datos
```

Ejemplo:

```python
dic_aeropuertos["1"] = {
    "nombre": "El Alto",
    "iata": "LPB",
    ...
}
```

Aquí todavía NO hay conexiones.

---

## Paso B — cargar rutas

Función:

```python
cargar_datos_rutas()
```

Lee:

```python
origen = fila[3]
destino = fila[5]
```

y añade:

```python
grafo_vuelos[origen].append(destino)
```

Aquí nace el grafo.

---

# 7. Grafo usado en BFS (Reto 1)

Función:

```python
calcular_alcance_bfs()
```

Usa el mismo grafo.

---

## Qué hace BFS aquí

Explora por niveles:

```text
Nivel 0 → aeropuerto origen
Nivel 1 → vuelos directos
Nivel 2 → vuelos con 1 escala
Nivel 3 → vuelos con 2 escalas
```

---

## Esquema visual

Supón:

```text
A → B → D
 \   \
  \   → E
   \
    → C
```

BFS desde A:

```text
Escala 0: A
Escala 1: B, C
Escala 2: D, E
```

Tu cola:

```python
cola = [(id_origen, 0)]
```

va expandiendo niveles.

---

# 8. Grafo usado en Kosaraju (Reto 2)

Aquí aparecen realmente DOS grafos.

---

# Grafo 1 — Original

```text
A → B
↑   ↓
D ← C
```

Representa rutas normales.

---

# Grafo 2 — Transpuesto

Función:

```python
construir_grafo_transpuesto()
```

Invierte todas las aristas.

---

## Ejemplo

Original:

```text
A → B
B → C
```

Transpuesto:

```text
A ← B
B ← C
```

o equivalentemente:

```text
B → A
C → B
```

---

# 9. Qué busca Kosaraju

Busca:

## Componentes Fuertemente Conexas (SCC)

Un grupo donde TODOS pueden llegar a TODOS.

---

## Ejemplo SCC

```text
A → B
↑   ↓
D ← C
```

Todos están conectados circularmente.

Entonces:

```text
SCC = {A, B, C, D}
```

---

# 10. Esquema completo del algoritmo Kosaraju

## Primera DFS

```python
dfs_orden_finalizacion()
```

Recorre el grafo original.

Va llenando:

```python
pila.append(u)
```

según terminan.

---

## Luego

Construyes el grafo invertido:

```python
grafo_transpuesto
```

---

## Segunda DFS

```python
dfs_scc()
```

Procesa nodos según la pila.

Cada DFS completa detecta un SCC.

---

# 11. Flujo visual general

## Construcción

```text
airports.dat
      ↓
 nodos

routes.dat
      ↓
 aristas

      ↓

GRAFO DIRIGIDO
```

---

## Reto 1

```text
Grafo
  ↓
 BFS
  ↓
 Aeropuertos alcanzables
```

---

## Reto 2

```text
Grafo original
      ↓
 DFS + pila

      ↓

Grafo transpuesto
      ↓
 DFS
      ↓
 SCCs
```

---

# 12. Idea intuitiva del proyecto

Tu programa modela:

```text
Aeropuerto = nodo
Vuelo = conexión
```

Entonces el mundo aéreo se convierte en una red.

Y los algoritmos responden preguntas como:

* ¿A dónde puedo llegar?
* ¿Qué grupos están aislados?
* ¿Qué aeropuertos forman comunidades?
* ¿Qué tan conectada está la red?

---

# 13. Resumen ultra corto

## Grafo principal

```text
Aeropuertos = nodos
Rutas = aristas dirigidas
```

---

## BFS

Explora por niveles.

Sirve para:

```text
alcance máximo con escalas
```

---

## Kosaraju

Usa:

* grafo original
* grafo invertido

para encontrar:

```text
componentes fuertemente conexas
```

---

# 14. Mini diagrama final

```text
          [ GRAFO DE VUELOS ]

      LPB ─────→ CBB
       ↑          │
       │          ↓
      VVI ←───── SUC


BFS:
LPB → qué tan lejos llego


Kosaraju:
¿Qué grupo queda conectado entre sí?
```
